<a href="https://colab.research.google.com/github/chiemahp/Flyrank-internship-ml/blob/main/capstone.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/chiemahp/Flyrank-internship-ml/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Question

*The research question and the decision it supports.*

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

QUESTION = (
    "Which pages look most likely to be declining with visible demand and should be reviewed for refresh?"
)
DECISION = "Use a ranked queue to support editor triage for refresh and review."

print(QUESTION)
print(DECISION)


Which pages look most likely to be declining with visible demand and should be reviewed for refresh?
Use a ranked queue to support editor triage for refresh and review.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

repo_root = Path.cwd().resolve()
for parent in [repo_root, *repo_root.parents]:
    candidate = parent / "data" / "raw" / "content_refresh_anonymized.csv"
    if candidate.exists():
        raw_path = candidate
        repo_root = parent
        break
else:
    raise FileNotFoundError("Could not locate the starter dataset.")

raw_df = pd.read_csv(raw_path)

numeric_cols = [
    "search_volume", "competition", "cpc", "word_count", "char_count",
    "impressions_90d", "clicks_90d", "pageviews_90d", "sessions_90d",
    "users_90d", "engaged_sessions_90d", "ai_sessions_90d",
    "scroll_events_90d", "days_with_impressions", "days_with_sessions",
    "impressions_last_30d", "clicks_last_30d", "sessions_last_30d",
    "impressions_prev_30d", "clicks_prev_30d", "sessions_prev_30d",
    "content_age_days", "age_tier_order", "days_since_last_update",
    "ctr", "avg_position", "engagement_rate", "scroll_rate",
    "ai_traffic_pct", "trend_pct",
]

categorical_cols = [
    "competition_level", "content_type", "main_intent", "provider_used",
    "model_used", "age_tier", "freshness_tier", "word_count_tier",
    "char_count_tier", "impression_tier", "position_tier", "trend_direction",
]

for col in numeric_cols:
    raw_df[col] = pd.to_numeric(raw_df[col], errors="coerce").fillna(0)
for col in categorical_cols:
    raw_df[col] = raw_df[col].fillna("unknown").astype(str).replace({"": "unknown", "nan": "unknown"})

prepared = raw_df[(raw_df["impressions_90d"] > 0) & (raw_df["content_age_days"] >= 90)].copy()
prepared["is_declining_label"] = prepared["trend_direction"].str.lower().eq("down").astype(int)
prepared["log_impressions_90d"] = np.log1p(prepared["impressions_90d"])
prepared["log_clicks_90d"] = np.log1p(prepared["clicks_90d"])
prepared["log_sessions_90d"] = np.log1p(prepared["sessions_90d"])
prepared["log_ai_sessions_90d"] = np.log1p(prepared["ai_sessions_90d"])
prepared["visibility_score"] = prepared["impressions_90d"].rank(pct=True)
prepared["freshness_risk_score"] = prepared["days_since_last_update"].rank(pct=True)
prepared["position_opportunity_score"] = (
    (1 - ((prepared["avg_position"].clip(lower=1, upper=50) - 1) / 49)).clip(0, 1) * prepared["visibility_score"]
)
prepared["baseline_refresh_score"] = (
    0.40 * prepared["visibility_score"]
    + 0.30 * prepared["freshness_risk_score"]
    + 0.30 * prepared["position_opportunity_score"]
).clip(0, 1)
prepared["review_priority"] = prepared["baseline_refresh_score"].rank(method="dense", ascending=False)

out_dir = repo_root / "work" / "outputs"
out_dir.mkdir(parents=True, exist_ok=True)

print(f"Rows in scored frame: {len(prepared):,}")
print(f"Declining-label rate: {prepared['is_declining_label'].mean():.3f}")
print(prepared[["content_id", "baseline_refresh_score", "review_priority", "is_declining_label"]].head(10).to_string(index=False))


Rows in scored frame: 30,000
Declining-label rate: 0.542
          content_id  baseline_refresh_score  review_priority  is_declining_label
content_304f48230142                0.586292           9282.0                   1
content_a1fb4e703a9e                0.732265           3726.0                   1
content_9aa793d4d895                0.536781          11163.0                   1
content_331d6c4de07b                0.775427           2257.0                   0
content_d99b7a2d90ca                0.445921          14921.0                   1
content_d4084a4bc775                0.599897           8767.0                   1
content_9a34b442b552                0.208194          24829.0                   1
content_a63219c6e95a                0.542702          10941.0                   0
content_5e6c160719bc                0.511805          12153.0                   1
content_c27558df2b0c                0.648108           6936.0                   1


## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score, precision_score, recall_score, f1_score
import pandas as pd
import numpy as np

feature_columns = [
    "search_volume", "competition", "cpc", "word_count", "char_count",
    "log_impressions_90d", "log_clicks_90d", "log_sessions_90d", "log_ai_sessions_90d",
    "days_with_impressions", "days_with_sessions", "content_age_days",
    "days_since_last_update", "ctr", "avg_position", "engagement_rate",
    "scroll_rate", "ai_traffic_pct", "competition_level", "content_type",
    "main_intent", "age_tier", "freshness_tier", "word_count_tier",
    "impression_tier", "position_tier",
]

X = pd.get_dummies(
    prepared[feature_columns],
    columns=["competition_level", "content_type", "main_intent", "age_tier", "freshness_tier", "word_count_tier", "impression_tier", "position_tier"],
    dummy_na=False,
)
y = prepared["is_declining_label"]

client_series = prepared.get("client_id", pd.Series(["unknown"] * len(prepared))).fillna("unknown").astype(str)
unique_clients = np.array(client_series.drop_duplicates())
rng = np.random.default_rng(42)
shuffled_clients = rng.permutation(unique_clients)
test_client_count = max(1, int(round(len(shuffled_clients) * 0.2)))
test_clients = set(shuffled_clients[:test_client_count])

mask = client_series.isin(test_clients).to_numpy()
train_index = prepared.index[~mask]
test_index = prepared.index[mask]

X_train = X.loc[train_index]
X_test = X.loc[test_index]
y_train = y.loc[train_index]
y_test = y.loc[test_index]

baseline_scores = prepared.loc[test_index, "baseline_refresh_score"].to_numpy()

model = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(class_weight="balanced", max_iter=1000, random_state=42)),
])
model.fit(X_train, y_train)
model_prob = model.predict_proba(X_test)[:, 1]
model_pred = (model_prob >= 0.5).astype(int)
baseline_pred = (baseline_scores >= 0.5).astype(int)

results = pd.DataFrame([
    {
        "model": "baseline_rules",
        "roc_auc": roc_auc_score(y_test, baseline_scores),
        "average_precision": average_precision_score(y_test, baseline_scores),
        "precision": precision_score(y_test, baseline_pred, zero_division=0),
        "recall": recall_score(y_test, baseline_pred, zero_division=0),
        "f1": f1_score(y_test, baseline_pred, zero_division=0),
    },
    {
        "model": "logistic_regression",
        "roc_auc": roc_auc_score(y_test, model_prob),
        "average_precision": average_precision_score(y_test, model_prob),
        "precision": precision_score(y_test, model_pred, zero_division=0),
        "recall": recall_score(y_test, model_pred, zero_division=0),
        "f1": f1_score(y_test, model_pred, zero_division=0),
    },
])

print(results.to_string(index=False))


              model  roc_auc  average_precision  precision   recall       f1
     baseline_rules 0.624027           0.466089   0.506925 0.201320 0.288189
logistic_regression 0.700291           0.521542   0.565934 0.566557 0.566245


## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*

In [ ]:
import json

summary = {
    "question": QUESTION,
    "decision": DECISION,
    "split_strategy": "client_holdout",
    "rows_scored": int(len(prepared)),
    "target_rate": round(float(prepared["is_declining_label"].mean()), 3),
    "metrics": results.to_dict(orient="records"),
}

(out_dir / "capstone_metrics.json").write_text(json.dumps(summary, indent=2))

print("Saved capstone metrics to", out_dir / "capstone_metrics.json")
print(results.to_string(index=False))


Saved capstone metrics to D:\Flyrank-internship-ml\work\outputs\capstone_metrics.json
              model  roc_auc  average_precision  precision   recall       f1
     baseline_rules 0.624027           0.466089   0.506925 0.201320 0.288189
logistic_regression 0.700291           0.521542   0.565934 0.566557 0.566245


In [ ]:
limitations = {
    "label_definition": "The target uses trend_direction as a proxy for decline, so it is directional rather than a verified business outcome.",
    "sample_scope": "Only pages with positive impressions and content age of at least 90 days were scored.",
    "validation": "The client holdout is a pragmatic estimate of future generalization, not a guarantee of live performance.",
    "causal_claim": "This notebook supports prioritization and review decisions; it does not prove that refreshing a page will improve business outcomes.",
}

for key, value in limitations.items():
    print(f"- {key}: {value}")


- label_definition: The target uses trend_direction as a proxy for decline, so it is directional rather than a verified business outcome.
- sample_scope: Only pages with positive impressions and content age of at least 90 days were scored.
- validation: The client holdout is a pragmatic estimate of future generalization, not a guarantee of live performance.
- causal_claim: This notebook supports prioritization and review decisions; it does not prove that refreshing a page will improve business outcomes.


## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*

In [ ]:
ranked = prepared.copy()
ranked["model_probability"] = np.nan
ranked.loc[test_index, "model_probability"] = model_prob
ranked["predicted_decline"] = np.nan
ranked.loc[test_index, "predicted_decline"] = model_pred
ranked["suggested_action"] = np.where(
    ranked["days_since_last_update"] >= 180,
    "refresh",
    np.where(ranked["ctr"] < 0.5, "review_ctr", "monitor"),
)

recommendations = ranked.sort_values(["model_probability", "baseline_refresh_score"], ascending=[False, False]).head(10)[[
    "content_id", "model_probability", "baseline_refresh_score", "suggested_action", "days_since_last_update", "avg_position", "ctr", "sessions_90d"
]]
recommendations.to_csv(out_dir / "capstone_recommendations.csv", index=False)

print(recommendations.to_string(index=False))


          content_id  model_probability  baseline_refresh_score suggested_action  days_since_last_update  avg_position  ctr  sessions_90d
content_8fdbff16a886           0.896852                0.581121       review_ctr                      20          12.0 0.00            18
content_e354f8e518c2           0.878528                0.424068       review_ctr                      20          23.4 0.00             3
content_da806b9f243e           0.878333                0.556123       review_ctr                      20           5.8 0.18             5
content_efe1a1a25ab0           0.873196                0.519376       review_ctr                      20          13.8 0.05             4
content_8f06ddfec8bc           0.869356                0.436511       review_ctr                      20          17.4 0.00             5
content_646483666d1d           0.867313                0.516536       review_ctr                      20          10.7 0.05             5
content_2ed4cd0eb5a9           0.8

In [ ]:
top_preview = recommendations.copy()
top_preview["model_probability"] = top_preview["model_probability"].round(3)
print("Top recommendation preview:")
print(top_preview.to_string(index=False))


Top recommendation preview:
          content_id  model_probability  baseline_refresh_score suggested_action  days_since_last_update  avg_position  ctr  sessions_90d
content_8fdbff16a886              0.897                0.581121       review_ctr                      20          12.0 0.00            18
content_e354f8e518c2              0.879                0.424068       review_ctr                      20          23.4 0.00             3
content_da806b9f243e              0.878                0.556123       review_ctr                      20           5.8 0.18             5
content_efe1a1a25ab0              0.873                0.519376       review_ctr                      20          13.8 0.05             4
content_8f06ddfec8bc              0.869                0.436511       review_ctr                      20          17.4 0.00             5
content_646483666d1d              0.867                0.516536       review_ctr                      20          10.7 0.05             5
conten

## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*

In [ ]:
import matplotlib.pyplot as plt

chart_dir = out_dir / "figures"
chart_dir.mkdir(parents=True, exist_ok=True)

action_counts = ranked["suggested_action"].value_counts().head(5)
fig, ax = plt.subplots(figsize=(7, 3.5))
action_counts.plot(kind="bar", ax=ax, color="#4C78A8")
ax.set_title("Suggested action mix")
ax.set_ylabel("Count")
fig.tight_layout()
fig_path = chart_dir / "capstone_action_mix.png"
fig.savefig(fig_path, dpi=150)
plt.close(fig)

print(f"Saved artifact to {fig_path}")
print(action_counts.to_string())


Saved artifact to D:\Flyrank-internship-ml\work\outputs\figures\capstone_action_mix.png
suggested_action
review_ctr    25615
monitor        4211
refresh         174
